In [12]:
import pandas as pd
import numpy as np
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from collections import defaultdict

In [13]:
def normalize(text):
    
    text=text.lower()
    
    text = re.sub(r'[^a-z0-9\s]', ' ', text) #normalize seperators to space
    text = re.sub(r'(\d+)\s*[-]?\s*kv', r' \1 ', text) #normalize all ###kv to be ###
    text = re.sub(r'l(\d+)',r'\1', text) #normalize all l### to be ###

    text = re.sub(r'([a-zA-Z]{2,})(\d)', r'\1 \2', text)#seperate letters from numbers with space
    text = re.sub(r'(\d)([a-zA-Z]{2,})', r'\1 \2', text)#seperate numbers from letters with space 
    
    text = re.sub(r'\s+', ' ', text).strip() #Collapse all white space

    #remove duplicate words to avoid over weighting
    words = text.split() 
    text = " ".join(dict.fromkeys(words))
    
    return text

In [14]:
def build_token_index(series_constraints):
    
    token_index=defaultdict(set)
    
    for idx, constraint in enumerate(series_constraints):
        tokens = constraint.split() #split each constrain into tokens
        
        for token in tokens:
            token_index[token].add(idx) #add the constraint index to the token index
            
    return token_index

In [15]:
def find_best_match(market_idx,market,word_importance,token_index):
    
    scores=defaultdict(set) #initialize the scores dictionary
    
    for token in market.iloc[i]["combined_norm"].split():
        if len(token) < 2 and token not in {"w", "e", "s", "n"} and not token.isdigit(): #skip low signal single characters
            continue
        idf_weight=word_importance[token] #get the importance of the token
        match_idxs=token_index[token] #get all indexes of matches from the token index
        if not match_idxs:
            continue
            
        for idx in match_idxs:
            if not scores[idx]: #if this is the first time the index is a match
                scores[idx] = idf_weight #the score of the matching index is the weight
            else:
                scores[idx] += idf_weight #otherwise, add the weight to the current score

    best_match_idx = max(scores, key=scores.get)
    
    return best_match_idx

In [16]:
#import data from dayzer, market, and pano
dayzer=pd.read_csv("Dayzer PJMISO constraint list.csv")
market=pd.read_csv("Market PJMISO constraint list.csv")
pano=pd.read_csv("Pano PJMISO constraint list.csv")

In [17]:
#normalize market constraint with contingency added for context
market["constraint_contingency"]=market["CONSTRAINT"].fillna("")+ " " + market["CONTINGENCY"].fillna("")
market["combined_norm"]=market["constraint_contingency"].apply(normalize)

#normalize dayzer
dayzer["constraint_norm"]=dayzer["NAME"].fillna("").apply(normalize)

#build pano constraint from facility + contingency and then normalize
pano["constraint"]=pano["Monitored Facility"].fillna("")+ " " + pano["Contingency Name"].fillna("")
pano["constraint_norm"]=pano["constraint"].fillna("").apply(normalize)

In [18]:
all_words=(market["combined_norm"].tolist()+dayzer["constraint_norm"].tolist()+pano["constraint_norm"].tolist())

vectorizer = TfidfVectorizer(token_pattern=r'(?u)\b\w+\b') #token pattern to include single character strings
x = vectorizer.fit_transform(all_words)
words = vectorizer.get_feature_names_out() #get the name of the words
idf_scores = vectorizer.idf_ #get the inverse document frequency for each word

word_importance = dict(zip(words, idf_scores)) #build a dictionary with key=word and value=idf

In [19]:
#build a dictionary with a key for each token in both dayzer and pano dataframe
#the value will be the index locations of those tokens in the respective dataframe
dayzer_token_index=build_token_index(dayzer["constraint_norm"])
pano_token_index=build_token_index(pano["constraint_norm"])

In [20]:
#initialize a dataframe for assignment output
output=market[["CONSTRAINT"]].rename(columns={"CONSTRAINT":"market_constraint"})
output["dayzer_constraint"]=""
output["pano_constraint"]=""

In [21]:
for i in range(len(market)):

    #find the dayzer constraint matching the market constraint
    idx_dayzer_match=find_best_match(i,market,word_importance,dayzer_token_index)
    dayzer_match=dayzer.loc[idx_dayzer_match,"NAME"]

    #find the pano constraint matching the market constraint
    idx_pano_match=find_best_match(i,market,word_importance,pano_token_index)
    pano_match=pano.loc[idx_pano_match,"constraint"]

    #store the dayzer and pano matches in the output dataframe
    output.loc[i,["dayzer_constraint","pano_constraint"]]=[dayzer_match,pano_match]

In [22]:
output.to_csv("Assignment 1 Output.csv",index=False)